# Phase 4: Build release tables

This notebook makes the release-table work inspectable. It joins Phase 2 articles and selected species–PMID matches and the full PubMed PMID index (so `n_articles` is not limited to the fetched sample) with Phase 3 gate decisions and the 73 domain-question answers, then writes versioned-input Parquet tables to `data/04_tables/release/`.

The transformations live in `pathogen_atlas_v2/tables/` and the orchestration in `pathogen_atlas_v2/pipelines/04_build_tables.py`. The first-evidence and time-to-publication metrics reproduce Version 1's emergence table logic. Nothing here turns Jev probabilities into pathogen labels beyond the configured 0.75 decision filters. Set `run_phase: true` in `configs/notebook_parameters.yaml` to build the release.

In [5]:
from pathlib import Path
import sys

from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pathogen_atlas_v2").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pathogen_atlas_v2 import notebook_utils as nbutils
from pathogen_atlas_v2.common.io import resolve_path
NOTEBOOK_CONFIG_PATH = "configs/notebook_parameters.yaml"
PARAMS = nbutils.notebook_phase_parameters("phase4_table_creation", NOTEBOOK_CONFIG_PATH)

import importlib
phase4_module = importlib.import_module("pathogen_atlas_v2.pipelines.04_build_tables")
phase4_preflight = phase4_module.phase4_preflight
build_phase4_tables = phase4_module.build_phase4_tables


## Inputs and processing steps

1. Load Phase 2 abstracts, selected-for-fetch species–PMID pairs, and the unsampled canonical PMID index for full PubMed article counts.
2. Load the full Phase 3 decision ledger for the configured run and apply the configured disease-evidence threshold and join article title, abstract, and publication year.
3. Load the cached domain expert probabilities for the configured run and derive yes/no answers at the configured question threshold. Keep only pairs that passed the disease-evidence gate.
4. Aggregate the Version 1 style full-search `n_articles`, accepted-evidence `n_causation`, and first-evidence year per species, plus the 73 feature summaries.
5. Reproduce emergence metrics: publication target year, years from first evidence to target, and evidence papers in the initial publication-year window.
6. Write Parquet tables, CSV web exports, and table metadata.

In [6]:
# Inspect whether Phase 1–3 releases needed by the joins are available.
preflight = phase4_preflight(PARAMS)
display(preflight)

# These are the functions used by the builder; each transformation is also
# independently callable with DataFrames for interactive analysis.
from pathogen_atlas_v2.tables.species_table import build_species_summary
from pathogen_atlas_v2.tables.feature_tables import build_feature_tables
from pathogen_atlas_v2.tables.emergence_tables import build_emergence_table
from pathogen_atlas_v2.tables.abstract_counts import build_abstract_count_table
from pathogen_atlas_v2.tables.web_exports import export_release_tables

display({
    "species_summary": build_species_summary.__doc__,
    "feature_tables": build_feature_tables.__doc__,
    "emergence_table": build_emergence_table.__doc__,
    "abstract_counts": build_abstract_count_table.__doc__,
    "web_exports": export_release_tables.__doc__,
})

{'status': 'ready',
 'inputs': {'abstracts': {'path': '/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version2/data/02_abstracts/release/abstracts.parquet',
   'exists': True},
  'species_matches': {'path': '/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version2/data/02_abstracts/release/abstract_species_matches.parquet',
   'exists': True},
  'canonical_pmid_index': {'path': '/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version2/data/02_abstracts/filtered/canonical_species_pmids.parquet',
   'exists': True},
  'sampled_pmid_index': {'path': '/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version2/data/02_abstracts/filtered/canonical_species_pmids_sampled.parquet',
   'exists': True},
  'gate_decisions': {'path'

{'species_summary': 'Summarize full PubMed search counts and accepted evidence per canonical species.\n\n    ``canonical_pmid_index`` provides the number of unique articles from the\n    complete query results; fetched abstracts are intentionally just a sample.\n    ',
 'feature_tables': 'Return a tidy answer table and a species-by-feature prevalence table.',
 'emergence_table': 'Build V1-like emergence metrics for species meeting configured bounds.',
 'abstract_counts': 'Count unique evidence abstracts by canonical species and publication year.',
 'web_exports': 'Write one UTF-8 CSV beside each Parquet release and return output paths.'}

In [7]:
if PARAMS.get("run_phase", False):
    phase4_result = build_phase4_tables(PARAMS)
else:
    phase4_result = {"status": "preflight_only", "reason": "Set run_phase: true to build the Phase 4 tables."}
display(phase4_result)

{'status': 'completed',
 'source': 'pathogen_atlas_v2_phase4',
 'run_id': 'jev-human-pathogen-evidence-v1',
 'expert_run_id': 'jev-domain-expert-features-v1',
 'confidence_threshold': 0.75,
 'model': 'typesafe/jev-1.14',
 'question_key': 'human_pathogen_evidence',
 'question_config_path': 'configs/llm_questions.yaml',
 'question_confidence_threshold': 0.75,
 'record_counts': {'evidence_records': 27031,
  'species_summary': 22799,
  'domain_expert_answers': 1973263,
  'species_feature_summary': 124319,
  'species_feature_matrix': 1703,
  'abstract_counts': 14398,
  'emergence': 541},
 'outputs': {'parquet_tables': {'evidence_records': '/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version2/data/04_tables/release/evidence_records.parquet',
   'species_summary': '/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version2/data/04_tables/release/species_sum

In [8]:
release_dir = resolve_path(PARAMS.get("release_dir", "data/04_tables/release"))
release_tables = {
    "evidence_records": release_dir / "evidence_records.parquet",
    "species_summary": release_dir / "species_summary.parquet",
    "domain_expert_answers": release_dir / "domain_expert_answers.parquet",
    "species_feature_summary": release_dir / "species_feature_summary.parquet",
    "species_feature_matrix": release_dir / "species_feature_matrix.parquet",
    "abstract_counts": release_dir / "abstract_counts.parquet",
    "emergence": release_dir / "emergence.parquet",
}
display(nbutils.existing_outputs(release_tables))
for name, path in release_tables.items():
    print(name)
    display(nbutils.read_table_if_exists(path, n_rows=5))

,name,path,exists,rows,size_bytes,content_type
0,evidence_records,/Users/f.costa/Documents/Projects/Repositories...,True,27031,18154965,parquet
1,species_summary,/Users/f.costa/Documents/Projects/Repositories...,True,22799,439287,parquet
2,domain_expert_answers,/Users/f.costa/Documents/Projects/Repositories...,True,1973263,5174488,parquet
3,species_feature_summary,/Users/f.costa/Documents/Projects/Repositories...,True,124319,179003,parquet
4,species_feature_matrix,/Users/f.costa/Documents/Projects/Repositories...,True,1703,179085,parquet
5,abstract_counts,/Users/f.costa/Documents/Projects/Repositories...,True,14398,48998,parquet
6,emergence,/Users/f.costa/Documents/Projects/Repositories...,True,541,31031,parquet


evidence_records


,pmid,canonical_species,matched_names,input_hash,provider,model,question_id,question_version,run_id,p_yes,...,error,completed_at,raw_response_path,confidence_threshold,accepted,title,abstract,publication_year,publication_date,doi
0,10028098,Abiotrophia defectiva,"[Abiotrophia defectiva, Streptococcus defectivus]",1b60e520b975d5639330f0960386f65dca8d4c355b11ab...,openrouter_jev,typesafe/jev-1.14,human_pathogen_evidence,human-pathogen-disease-attribution-v1,jev-human-pathogen-evidence-v1,0.98,...,None,2026-10-30T08:32:50+00:00,/Users/f.costa/Documents/Projects/Repositories...,0.75,True,Iatrogenic meningitis due to Abiotrophia defec...,,1999,1999 Jan,10.1086/517189
1,10203522,Abiotrophia defectiva,"[Abiotrophia defectiva, Streptococcus defectivus]",c7824fb68117ec01d2c45c76f65a901f6bbc4022313a0c...,openrouter_jev,typesafe/jev-1.14,human_pathogen_evidence,human-pathogen-disease-attribution-v1,jev-human-pathogen-evidence-v1,0.99,...,None,2026-10-30T08:32:50+00:00,/Users/f.costa/Documents/Projects/Repositories...,0.75,True,Abiotrophia species as a cause of endophthalmi...,"Microorganisms of the genus Abiotrophia, membe...",1999,1999 May,10.1128/JCM.37.5.1564-1566.1999
2,10629690,Abiotrophia defectiva,"[Abiotrophia defectiva, Streptococcus defectivus]",67100116f05e55f43ab067a30a568daf2291a02ad69a1c...,openrouter_jev,typesafe/jev-1.14,human_pathogen_evidence,human-pathogen-disease-attribution-v1,jev-human-pathogen-evidence-v1,0.98,...,None,2026-10-30T08:32:51+00:00,/Users/f.costa/Documents/Projects/Repositories...,0.75,True,[Abiotrophia related endocarditis: contributio...,BACKGROUND: Former nutritionally-deficient (va...,1999,1999 Dec 11,
3,10681362,Abiotrophia defectiva,"[Abiotrophia defectiva, Streptococcus defectivus]",d346de61926a80cf30fa83ef44027420ee9f269999bbea...,openrouter_jev,typesafe/jev-1.14,human_pathogen_evidence,human-pathogen-disease-attribution-v1,jev-human-pathogen-evidence-v1,0.99,...,None,2026-10-30T08:32:51+00:00,/Users/f.costa/Documents/Projects/Repositories...,0.75,True,Characterization of the Tn916-like transposon ...,Clinical blood isolates from three sequential ...,2000,2000 Mar,10.1128/AAC.44.3.790-793.2000
4,11095068,Abiotrophia defectiva,"[Abiotrophia defectiva, Streptococcus defectivus]",71d7dc13a43938e0816ddf7cd545e617cf772f6943ca2c...,openrouter_jev,typesafe/jev-1.14,human_pathogen_evidence,human-pathogen-disease-attribution-v1,jev-human-pathogen-evidence-v1,0.98,...,None,2026-10-30T08:32:51+00:00,/Users/f.costa/Documents/Projects/Repositories...,0.75,True,Infiltrative keratitis associated with extende...,PURPOSE: Infiltrative keratitis is a common co...,2000,2000 Nov,10.1097/00003226-200011000-00024


species_summary


,canonical_species,n_articles,n_sampled,n_causation,first_evidence_year,domain,kingdom,phylum,class,order,family,genus,source
0,Abditibacterium utsteinense,0,0,0,<NA>,Bacteria,Pseudomonadati,Abditibacteriota,Abditibacteriia,Abditibacteriales,Abditibacteriaceae,Abditibacterium,lpsn
1,Abiotrophia defectiva,184,184,149,1995,Bacteria,Bacillati,Bacillota,Bacilli,Lactobacillales,Aerococcaceae,Abiotrophia,lpsn
2,Absicoccus intestinalis,0,0,0,<NA>,Bacteria,Bacillati,Bacillota,Erysipelotrichia,Erysipelotrichales,Erysipelotrichaceae,Absicoccus,lpsn
3,Absicoccus porci,0,0,0,<NA>,Bacteria,Bacillati,Bacillota,Erysipelotrichia,Erysipelotrichales,Erysipelotrichaceae,Absicoccus,lpsn
4,Abyssalbus ytuae,0,0,0,<NA>,Bacteria,Pseudomonadati,Bacteroidota,Flavobacteriia,Flavobacteriales,Flavobacteriaceae,Abyssalbus,lpsn


...

,canonical_species,feature_air,feature_airborne,feature_alcohol,feature_anaerobic,feature_animal_contact,feature_aquatic_vertebrates,feature_asymptomatic_carriage,feature_bacteraemia,feature_biofilm,...,feature_systemic,feature_temperature,feature_toxins,feature_urinary,feature_vaccine,feature_vectorborne,feature_vertical,feature_waterborne,feature_wound,feature_zoonosis
0,Abiotrophia defectiva,0.0,0.0,0.0,0.006711,0.0,0.0,0.0,0.288591,0.013423,...,0.114094,0.026846,0.0,0.026846,0.0,0.0,0.0,0.0,0.013423,0.0
1,Acetobacter cibinongensis,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Acetobacter ghanensis,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Acetobacter indonesiensis,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.333333,0.0,...,0.333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Acetobacter tropicalis,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


abstract_counts


,canonical_species,publication_year,n_abstracts
0,Abiotrophia defectiva,1995,1
1,Abiotrophia defectiva,1997,1
2,Abiotrophia defectiva,1999,3
3,Abiotrophia defectiva,2000,3
4,Abiotrophia defectiva,2002,2


emergence


,canonical_species,n_articles,n_sampled,n_causation,first_evidence_year,domain,kingdom,phylum,class,order,family,genus,source,year_reaching_n_pubs,pubs_in_first_year_window,n_years_to_n_pubs
0,Helicobacter pylori,15644,200,74,2014,Bacteria,Pseudomonadati,Pseudomonadota,Epsilonproteobacteria,Campylobacterales,Helicobacteraceae,Helicobacter,lpsn,2014,64,0
1,Escherichia coli,20529,200,58,2018,Bacteria,Pseudomonadati,Pseudomonadota,Gammaproteobacteria,Enterobacterales,Enterobacteriaceae,Escherichia,lpsn,2018,58,0
2,Burkholderia cenocepacia,644,200,79,2004,Bacteria,Pseudomonadati,Pseudomonadota,Betaproteobacteria,Burkholderiales,Burkholderiaceae,Burkholderia,lpsn,2004,48,0
3,Streptococcus pneumoniae,16927,200,78,2004,Bacteria,Bacillati,Bacillota,Bacilli,Lactobacillales,Streptococcaceae,Streptococcus,lpsn,2004,38,0
4,Clostridioides difficile,16119,200,108,1992,Bacteria,Bacillati,Bacillota,Clostridia,Peptostreptococcales,Peptostreptococcaceae,Clostridioides,lpsn,1992,20,0
